# 13.1 언어모델링: 다음 토큰 예측과 사전학습·파인튜닝 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter13_1_next_token_lm.ipynb)

책 본문: [13.1 언어모델링: 다음 토큰 예측과 사전학습·파인튜닝](https://smhanlab.com/book-ml/kor/ml1/chapter13/1.html)

이 노트북은 책 13.1절의 모든 수치를 **실제로 실행해서** 검증합니다:

1. **BPE** — 25글자 corpus에서 빈도 기반 병합 6회 후 토큰 수·어휘 사전 크기
2. **n-gram** — bigram `next_token_distribution`("서울의" → 0.5/0.5), unigram PPL vs bigram PPL
3. **softmax** — \(z=(4,1,0,-3)\) → (0.935, 0.047, 0.017, 0.001) + shift-invariance
4. **온도** — \(T=0.5,1,2,10\)의 softmax 분포 비교(점근 → 평탄), 시각화
5. **교차 엔트로피 & PPL** — \(-\log p\) 세 값, PPL = \(e^{\text{평균손실}}\)
6. **미니 신경망 언어모델** — torch로 다음 토큰 예측 학습, 손실 감소 곡선

In [1]:
import math
from collections import defaultdict, Counter

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
print("numpy", np.__version__)

numpy 2.4.6


## 1. BPE: 빈도 기반 병합으로 자연스러운 토큰 단위 발견

본문 예 — 한국어 문장 세 줄("서울의 겨울은 춥다 / 서울의 봄은 따뜻하다 /
파리의 겨울은 춥다")을 **여백 없이** 문자 단위로 토큰화하면 25개의 글자
(서로 다른 13종)가 됩니다. "가장 자주 함께 나타나는 인접 쌍"을 반복
병합하면, **반복되는 조각(서울의, 겨울은)이 먼저** 묶여야 합니다.
(빈도가 같으면 `max`가 처음 만난 쌍을 택하므로 결정적입니다.)

In [2]:
def bpe(text, merges, verbose=True):
    """가장 빈번한 인접 쌍을 반복 병합하는 가장 단순한 BPE.
    returns: (최종 토큰 리스트, 최종 어휘 사전 set)"""
    toks = [c for c in text if c != ' ']
    vocab = set(toks)
    history = []
    for m in range(merges):
        pairs = {}
        for i in range(len(toks) - 1):
            p = (toks[i], toks[i + 1])
            pairs[p] = pairs.get(p, 0) + 1
        if not pairs:
            break
        best = max(pairs.items(), key=lambda kv: (kv[1], -list(pairs).index(kv[0])))[0]
        new = best[0] + best[1]
        vocab.add(new)
        merged, i = [], 0
        while i < len(toks):
            if i < len(toks) - 1 and (toks[i], toks[i + 1]) == best:
                merged.append(new); i += 2
            else:
                merged.append(toks[i]); i += 1
        toks = merged
        if verbose:
            history.append((m + 1, best[0], best[1], new, pairs[best], len(toks), len(vocab)))
            print(f"merge{m+1}: '{best[0]}'+'{best[1]}' -> '{new}'  (빈도 {pairs[best]})  토큰 {len(toks)} 어휘 {len(vocab)}")
    return toks, vocab

text = "서울의겨울은춥다서울의봄은따뜻하다파리의겨울은춥다"
print("시작: 글자 수 =", len([c for c in text if c != ' ']), " 서로 다른 글자 =", len(set(text)))
toks, vocab = bpe(text, merges=6)
print("최종 토큰:", toks)
assert len(toks) == 13, len(toks)          # 본문 표: 6회 병합 후 13개 토큰
assert len(vocab) == 19, len(vocab)        # 본문 표: 어휘 사전 19
assert "서울의" in vocab and "겨울은" in vocab   # 반복 조각이 먼저 묶임
print("-> 본문 BPE 표(25글자->13토큰, 어휘 13->19)와 정확히 일치!")

시작: 글자 수 = 25  서로 다른 글자 = 13
merge1: '서'+'울' -> '서울'  (빈도 2)  토큰 23 어휘 14
merge2: '서울'+'의' -> '서울의'  (빈도 2)  토큰 21 어휘 15
merge3: '겨'+'울' -> '겨울'  (빈도 2)  토큰 19 어휘 16
merge4: '겨울'+'은' -> '겨울은'  (빈도 2)  토큰 17 어휘 17
merge5: '겨울은'+'춥' -> '겨울은춥'  (빈도 2)  토큰 15 어휘 18
merge6: '겨울은춥'+'다' -> '겨울은춥다'  (빈도 2)  토큰 13 어휘 19
최종 토큰: ['서울의', '겨울은춥다', '서울의', '봄', '은', '따', '뜻', '하', '다', '파', '리', '의', '겨울은춥다']
-> 본문 BPE 표(25글자->13토큰, 어휘 13->19)와 정확히 일치!


## 2. n-gram 언어모델: 다음 토큰 분포를 손으로 세기

본문 `next_token_distribution`을 그대로 실행해, bigram 모델이
"서울의" 뒤 분포를 (겨울은 0.5, 봄은 0.5)로 세는지 확인합니다.
이어서 **unigram PPL**과 **bigram PPL**을 계산해 "문맥을 쓸수록
PPL이 낮아진다"를 숫자로 확인합니다. (단어 단위 토큰 — `.split()`)

In [3]:
def next_token_distribution(corpus_tokens, context_word):
    # context_word가 등장한 모든 위치에서, 바로 다음 토큰을 세어
    # {다음단어: 확률} 딕셔너리로 반환 (bigram 모델, n=2)
    counts = defaultdict(int)
    for i in range(len(corpus_tokens) - 1):
        if corpus_tokens[i] == context_word:
            counts[corpus_tokens[i + 1]] += 1
    total = sum(counts.values())
    if total == 0:
        return {}   # 문맥이 한 번도 등장한 적 없으면 예측 근거 없음
    return {w: c / total for w, c in counts.items()}

corpus = "서울의 겨울은 춥다 서울의 봄은 따뜻하다 파리의 겨울은 춥다".split()
dist = next_token_distribution(corpus, "서울의")
print("next_token_distribution(corpus, '서울의') =", dist)
assert dist == {'겨울은': 0.5, '봄은': 0.5}, dist   # 본문과 일치
assert next_token_distribution(corpus, "뉴욕의") == {}   # OOV -> 빈 딕셔너리
print("OOV ('뉴욕의') ->", next_token_distribution(corpus, '뉴욕의'), "  (빈 딕셔너리)")

# --- unigram PPL (n=1, 문맥 무시, 전체 N개 토큰) ---
N = len(corpus)
cnt = Counter(corpus)
ppl_uni = math.exp(-sum(math.log(cnt[t] / N) for t in corpus) / N)

# --- bigram PPL (n=2, N-1개 예측 가능 위치) ---
bcnt = Counter((corpus[i], corpus[i + 1]) for i in range(N - 1))
ccount = Counter(corpus[:-1])
ppl_bi = math.exp(-sum(math.log(bcnt[(corpus[i], corpus[i + 1])] / ccount[corpus[i]])
                       for i in range(N - 1)) / (N - 1))

print(f"unigram PPL = {ppl_uni:.3f}")
print(f"bigram  PPL = {ppl_bi:.3f}")
assert abs(ppl_uni - 5.67) < 0.05, ppl_uni   # 본문: unigram PPL ~ 5.67
assert abs(ppl_bi - 1.19) < 0.05, ppl_bi     # 본문: bigram PPL ~ 1.19
assert ppl_bi < ppl_uni
print("-> 문맥을 보는 bigram(1.19)이 문맥을 무시하는 unigram(5.67)보다 PPL이 낮다. 확인!")

next_token_distribution(corpus, '서울의') = {'겨울은': 0.5, '봄은': 0.5}
OOV ('뉴욕의') -> {}   (빈 딕셔너리)
unigram PPL = 5.670
bigram  PPL = 1.189
-> 문맥을 보는 bigram(1.19)이 문맥을 무시하는 unigram(5.67)보다 PPL이 낮다. 확인!


## 3. softmax: 원점수를 확률로 (+ shift-invariance)

본문 예 — 어휘 4개, 원점수 \(z = (4, 1, 0, -3)\). softmax를 적용하면
첫 번째 토큰에 0.935가 모아야 합니다. 또 **모든 logit에 같은 상수를
더해도 분포가 변하지 않는다**(shift-invariance)는 성질도 확인합니다.

In [4]:
def softmax(z):
    m = max(z)                      # 수치 안정: 최대값 먼저 빼기
    ex = [math.exp(v - m) for v in z]
    s = sum(ex)
    return [e / s for e in ex]

z = [4.0, 1.0, 0.0, -3.0]
p = softmax(z)
print("softmax(z) =", [f"{x:.4f}" for x in p])
assert abs(p[0] - 0.935) < 5e-4, p[0]
assert abs(p[1] - 0.047) < 5e-4
assert abs(p[2] - 0.017) < 5e-4
assert abs(sum(p) - 1.0) < 1e-12     # 합이 1
# 모든 logit에 +100를 더해도 분포 불변 (shift-invariant)
p_shift = softmax([v + 100 for v in z])
assert all(abs(a - b) < 1e-12 for a, b in zip(p, p_shift))
print("shift-invariance 확인: 모든 logit에 +100를 더해도 분포가 정확히 동일.")
# 원점수 차이가 확률 비율로: z0-z3 = 7 -> P0/P3 = e^7
print(f"P[0]/P[3] = {p[0]/p[3]:.0f}  (e^7 = {math.exp(7):.0f})  -> logit 차 7이 확률 비율 e^7로")

softmax(z) = ['0.9354', '0.0466', '0.0171', '0.0009']
shift-invariance 확인: 모든 logit에 +100를 더해도 분포가 정확히 동일.
P[0]/P[3] = 1097  (e^7 = 1097)  -> logit 차 7이 확률 비율 e^7로


## 4. 온도(Temperature): 확신도를 조절하는 손

같은 \(z=(4,1,0,-3)\)에서 \(T=0.5, 1.0, 2.0, 10.0\)로 softmax를
돌려 분포가 **점근(peaky) → 평탄(flat)** 해지는 것을 그립니다.
이것이 실전 LLM의 `temperature` 파라미터 원리입니다.

In [5]:
temps = [0.5, 1.0, 2.0, 10.0]
labels = ["token 1", "token 2", "token 3", "token 4"]
probs_by_T = {T: softmax([v / T for v in z]) for T in temps}

for T in temps:
    print(f"T={T:4}: " + "  ".join(f"{x:.4f}" for x in probs_by_T[T]))
assert abs(probs_by_T[0.5][0] - 0.997) < 5e-4
assert abs(probs_by_T[1.0][0] - 0.935) < 5e-4
assert abs(probs_by_T[2.0][0] - 0.720) < 5e-4
assert abs(probs_by_T[10.0][0] - 0.344) < 5e-4
# T가 커질수록 최고 확률(=확신도)은 감소해야 함
peak = [max(probs_by_T[T]) for T in temps]
print("최고 확률(확신도) 변화:", [f"{x:.3f}" for x in peak])
assert all(peak[i] >= peak[i + 1] for i in range(len(peak) - 1))

x = np.arange(len(labels))
width = 0.2
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, T in enumerate(temps):
    off = (i - 1.5) * width
    bars = ax.bar(x + off, probs_by_T[T], width, label=f"T = {T}")
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Next-token probability")
ax.set_title("Softmax distribution vs. temperature (T) — lower T peaks, higher T flattens")
ax.set_ylim(0, 1.08)
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{IMG}/ch13_1_temperature_softmax.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch13_1_temperature_softmax.svg")

T= 0.5: 0.9972  0.0025  0.0003  0.0000
T= 1.0: 0.9354  0.0466  0.0171  0.0009
T= 2.0: 0.7201  0.1607  0.0975  0.0217
T=10.0: 0.3439  0.2548  0.2305  0.1708
최고 확률(확신도) 변화: ['0.997', '0.935', '0.720', '0.344']
저장: /home/smhan/book-ml/kor/src/images/ch13_1_temperature_softmax.svg


## 5. 교차 엔트로피 손실 & 퍼플렉시티

한 토큰 손실 \(-\log p\)의 세 가지 대표 값(0.9 / 0.5 / 0.001)을
계산하고, **PPL = \(e^{\text{평균 손실}}\)** 이 성립함을 확인합니다.
"확률이 높을수록 손실이 작아진다"는 것이 숫자로 드러나야 합니다.

In [6]:
def cross_entropy_loss(probs, true_idx):
    return -math.log(probs[true_idx])

def perplexity(token_losses):
    return math.exp(sum(token_losses) / len(token_losses))

for pval in (0.9, 0.5, 0.001):
    print(f"-log({pval:<6}) = {-math.log(pval):.3f}")
assert abs(-math.log(0.9) - 0.105) < 5e-4
assert abs(-math.log(0.5) - 0.693) < 5e-4
assert abs(-math.log(0.001) - 6.908) < 5e-4

# 본문 연습문제3: softmax 예의 정답(인덱스0, p=0.935) 손실
print(f"정답 p=0.935 -> 손실 = {-math.log(0.935):.3f}")
assert abs(-math.log(0.935) - 0.067) < 5e-4

# PPL = exp(평균 손실)
tok_losses = [0.067, 0.693, 3.912]
ppl = perplexity(tok_losses)
print(f"PPL([0.067, 0.693, 3.912]) = {ppl:.2f}")
assert abs(ppl - 4.75) < 0.05, ppl
# 손실 감소 -> PPL 감소 (모노톤)
assert perplexity([0.1, 0.1, 0.1]) < perplexity([3.9, 3.9, 3.9])
print("-> 손실이 작을수록 PPL이 낮다(모노톤). 확인!")

-log(0.9   ) = 0.105
-log(0.5   ) = 0.693
-log(0.001 ) = 6.908
정답 p=0.935 -> 손실 = 0.067
PPL([0.067, 0.693, 3.912]) = 4.75
-> 손실이 작을수록 PPL이 낮다(모노톤). 확인!


## 6. 미니 신경망 언어모델: 다음 토큰 예측 = 분포 학습

앞의 n-gram(수동 세기)을 **학습으로** 대체하는 최소 실험입니다.
6개 단어 어휘, embedding(8차원) + 선형 출력층의 아주 작은 모델로
같은 corpus를 다음 토큰 예측 학습시킵니다. 손실이 **단순한
초기값에서 계속 감소**하는 것을 학습 곡선으로 확인합니다 —
"다음 토큰 예측 손실을 최소화하는 것"이 곧 "문장 분포 학습"인
것이 수치로 보입니다. (torch CPU, seed=0)

In [7]:
import torch
import torch.nn as nn
torch.manual_seed(0)

words = ["서울의", "겨울은", "춥다", "봄은", "따뜻하다", "파리의"]
w2i = {w: i for i, w in enumerate(words)}
line = "서울의 겨울은 춥다 서울의 봄은 따뜻하다 파리의 겨울은 춥다".split()
seq = line * 50                                   # 반복해서 mini-batch 대신 단일 배치로 학습
inputs  = torch.tensor([w2i[t] for t in seq[:-1]])
targets = torch.tensor([w2i[t] for t in seq[1:]])

model = nn.Sequential(nn.Embedding(len(words), 8), nn.Linear(8, len(words)))
opt = torch.optim.Adam(model.parameters(), lr=0.05)
lossf = nn.CrossEntropyLoss()

losses = []
for epoch in range(300):
    opt.zero_grad()
    loss = lossf(model(inputs), targets)
    loss.backward()
    opt.step()
    losses.append(loss.item())

print("epoch 0:   loss =", f"{losses[0]:.3f}", " (초기, 대략 log(6)=" + f"{math.log(6):.3f}" + " 부근)")
print("epoch 99:  loss =", f"{losses[99]:.3f}")
print("epoch 199: loss =", f"{losses[199]:.3f}")
print("epoch 299: loss =", f"{losses[-1]:.4f}", " -> PPL =", f"{math.exp(losses[-1]):.3f}")
assert losses[-1] < losses[0]                     # 손실 감소
assert losses[-1] < 0.5                            # 수렴
assert all(losses[i] >= losses[i + 1] - 1e-3 for i in range(0, 299, 30))  # 거의 모노톤 감소

# 학습된 모델이 bigram과 거의 같은 PPL을 내는가?
print(f"학습 미니LM PPL = {math.exp(losses[-1]):.3f}  (vs 손 bigram PPL = {ppl_bi:.3f})")

ep = np.arange(1, len(losses) + 1)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ep, losses, color="#4878a8", lw=2)
ax.axhline(math.log(6), color="#d95f02", ls="--", lw=1, label="log(6) = uniform-distribution bound")
ax.set_xlabel("epoch"); ax.set_ylabel("Next-token-prediction cross-entropy")
ax.set_title("Mini neural LM training curve — next-token prediction = learning a distribution")
ax.set_yscale("log")
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout()
fig.savefig(f"{IMG}/ch13_1_lm_learning_curve.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch13_1_lm_learning_curve.svg")

epoch 0:   loss = 2.000  (초기, 대략 log(6)=1.792 부근)
epoch 99:  loss = 0.155
epoch 199: loss = 0.154
epoch 299: loss = 0.1545  -> PPL = 1.167
학습 미니LM PPL = 1.167  (vs 손 bigram PPL = 1.189)
저장: /home/smhan/book-ml/kor/src/images/ch13_1_lm_learning_curve.svg


## 7. 정리

| 실험 | 관측 | 본문의 어떤 주장을 확인하나 |
|---|---|---|
| BPE 6회 병합 | 25글자→13토큰, `서울의`/`겨울은`이 먼저 묶임 | 빈도 통계만으로 자연스러운 토큰 단위 발견 |
| bigram `next_token_distribution` | "서울의"→(겨울은 0.5, 봄은 0.5), OOV→`{}` | 다음 토큰 분포를 세는 것; 과소표본·OOV 한계 |
| unigram vs bigram PPL | 5.67 vs 1.19 | 문맥을 쓸수록 PPL이 낮아진다 |
| softmax(4,1,0,-3) | (0.935, 0.047, 0.017, 0.001), +100 불변 | 원점수→확률, logit 차=확률 비율, shift-invariance |
| 온도 스윕 | T↓ 점근 / T↑ 평탄 | `temperature`가 확신도를 조절 |
| 교차 엔트로피 & PPL | 0.9→0.105, 0.001→6.908; PPL=e^손실 | 확률↑=손실↓, PPL=선택지 헤매기 척도 |
| 미니LM 학습 | loss 2.0→0.15 (PPL 1.17) | 다음 토큰 예측 손실 최소화 = 분포 학습 |

**다음 13.2절**: 이 "다음 토큰 예측" 모델을 **가중치 없이** 프롬프트
(입력)만으로 제어하는 **프롬프팅**(zero-shot, few-shot,
chain-of-thought)을 다룹니다.